In [ ]:
import matplotlib.pyplot as plt
%matplotlib widget

import numpy as np

<figure id="figure-3">
<div style="background-color: white">

![](figures/dct.svg)</div>

<figcaption style="text-align: center">Figure 3: A DCT can be treated as an N channel filter bank where the coefficients of the
filters are the basis functions.</figcaption></figure>


# 7 The Discrete Cosine Transform (DCT)

The DCT is a method of performing energy compaction that is rather different
from the pyramid method. It operates on non-overlapping blocks of pixels
(typically $8 \times 8$ pixels in size) by a reversible linear transform
process, such that each block of pixels is replaced by a block of the same
number of transform coefficients. If all the transform coefficients for a
given block are transmitted unaltered to the decoder, then the original block
of pixels can be exactly recovered by the inverse transform process.

In practise the transform coefficients are quantised before transmission, and
if energy compaction has occurred, then fewer bits will be needed to send the
coefficients than the original pixels. A key advantage of transform-based
methods is that there is no expansion of the number of samples (the
transformed block is the same size as the original block of pixels), whereas
the previous pyramid method expands the data by
$1 + \frac{1}{4} + \frac{1}{16} + \ldots \approx 1.33$ times, which is not very desirable for data
compression.

> Note that currently the answer to qns in this notebook is made by user/AI, may not be completely verifiable


## 7.1 Definition of the DCT

The one-dimensional form of the DCT is closely related to the Discrete Fourier
Transform (DFT). The 1-D $N$-point DCT is defined as follows:

$$
y(k) = \sum_{n=0}^{N-1} C_{kn}\ x(n) \quad \text{for} \quad 0 \le k \le N-1 \\
  \text{where }\quad C_{0n} = \sqrt{\frac{1}{N}}  \\
    \text{and } \quad C_{kn} = \sqrt{\frac{2}{N}}\ \cos
\frac{k(n+\frac{1}{2})\pi}{N} \quad \text{for} \quad 1 \le k \le N-1
$$

The equivalent inverse DCT is:

$$
x(n) = \sum_{k=0}^{N-1} C_{kn}\ y(k) \quad \text{for} \quad 0 \le n \le N-1 \\
 \text{where $C_{kn}$ is defined as above.}\\
$$

(This is actually the Type-II DCT, and the inverse is the Type-III DCT - other types have slightly different relative phases})

We see that the forward transform is equivalent to multiplication of the
$N$-point column vector $[x(0) \ldots x(N-1)]'$ by an $N \times N$ matrix,
containing $C_{kn}$ at each location $(k,n)$, to produce the $N$-point column
vector $[y(0) \ldots y(N-1)]'$. Similarly the inverse transform is equivalent
to multiplication of the $y$ vector by the transpose of the $C$ matrix to give
the $x$ vector. In python3 + numpy notation these become:

`y = C @ x` and `x = C.T @ y`

Note that C is an **orthonormal matrix since its inverse is just its
transpose** (its rows are othogonal to each other and have unit energy).

The two-dimensional version of the DCT (as used for image compression) is a
simple extension of the above 1-D DCT. For an $N \times N$ block of pixels,
the $N$-point 1-D DCT is first applied to each column of the block to give $N$
columns of coefficients. Then the same 1-D DCT is applied to the rows of
these coefficients to give the 2-D transform coefficients.

In python3 + numpy notation, if the input block of pixels is matrix X, the output
block of 2-D transformed coefficients Y is given by:

`Y = (C @ (C @ X).T).T` or more simply `Y = C @ X @ C.T`

where C is the 1-D transform matrix as above. Note that in the 2-D
transform, it does not matter whether the rows or the columns are transformed
first (because the transform is linear and separable).


## 7.2 Applying the DCT to images

Conceptually the 2-D DCT is applied to all non-overlapping $N \times N$ blocks
of pixels in an image (we assume that the image dimensions are exact multiples
of $N$). However it is simplest and most efficient to perform 1-D
$N$-point DCTs on all the columns of the image first, and then repeat the
operation on the transpose of the result to transform the rows.

**First generate an 8-point 1-D Type-II DCT matrix C8**


In [ ]:
from cued_sf2_lab.dct import dct_ii

C8 = dct_ii(8)

Take a look at the function `dct_ii` and list `C8` to check
that it agrees with the definitions for $C_{kn}$ given above:


In [ ]:
import inspect
import IPython.display
IPython.display.Code(inspect.getsource(dct_ii), language="python")

**Plot the rows of `C8` using `plot(C8.T)`.**


In [ ]:
fig, ax = plt.subplots(figsize=(4, 3))
ax.plot(C8.T);
ax.set(xlabel='Sample index $n$', ylabel='$C_{kn}$', title='Rows of C8 (1-D basis vectors)')
ax.legend([f'$k={k}$' for k in range(8)], fontsize=6, ncol=2)
fig.tight_layout()
fig.savefig('zach/images/dct_1d_basis.png', dpi=150, bbox_inches='tight')

When we calculate the 1-D transform of an 8-point block of data, each
transform coefficient represents the component of the data that is
correlated with the corresponding row of `C8`. Hence the
first coefficient represents the dc component, the second one
represents the approximate average slope, and so on. The later
coefficients represent progressively higher frequency components
in the data.

The function `colxfm(X, C8)` will perform a 1-D transform on
the columns of image `X` using `C8`. We can
therefore perform a 2-D transform on `X` by using `colxfm` twice, once with transpose operators, as follows:


In [ ]:
from cued_sf2_lab.familiarisation import load_mat_img
from cued_sf2_lab.dct import colxfm

X_pre_zero_mean, cmaps_dict = load_mat_img(img='lighthouse.mat', img_info='X', cmap_info={'map', 'map2'})
X = X_pre_zero_mean - 128.0

Y = colxfm(colxfm(X, C8).T, C8).T

In `Y`, each $8 \times 8$ block of pixels has been replaced by an
equivalent block of transform coefficients. The coefficient in the top left
corner of each block represents the dc value of the block of pixels;
coefficients along the top row represent increasing horizontal frequency
components, and along the left column represent increasing vertical frequency
components. Other coefficients represent various combinations of horizontal
and vertical frequencies, in proportion to their horizontal and vertical
distances from the top left corner.

If we try to display `Y` directly as an image, it is rather
confusing because the different frequency components of each block
are all present adjacent to each other.


In [ ]:
from cued_sf2_lab.familiarisation import plot_image

fig, ax = plt.subplots()
plot_image(Y, ax=ax);

A much more meaningful
image is created if we group all the coefficients of a given type
together into a small sub-image, and display the result as an $8
\times 8$ group of sub-images, one for each coefficient type. The
function `regroup(Y, N)` achieves this regrouping, where $N$ is
the size of the original transform blocks. You need to ensure that **X has zero mean (by subtracting 128)** before you start transforming it, otherwise the dc coefficient will be purely positive, whereas the
others are symmetrically distributed about zero. Also, an $N
\times N$ 2-D DCT introduces a gain factor of $N$ in order to
preserve constant total energy between the pixel and transform
domains: we need to divide by $N$ _when displaying_ to get back to the expected range.

> so after we do the regular dct on each 8x8 block. we will end up with subimages of 8x8 size each. regrouping will group all the similar coefficients together, so we get 64 (8x8 groups), one group for each coefficient. Each group is of size 256//8=32.
> so the top left DC component is just the meanpool of each 8x8 block

Hence we can display `Y` meaningfully using:


In [ ]:
from cued_sf2_lab.dct import regroup

N = 8
fig, ax = plt.subplots()
plot_image(regroup(Y, N)/N, ax=ax);

In this image, you should see a small replica of the original in the top left
corner (the dc coefficients), and other sub-images showing various edges from
the original, representing progressively higher frequencies as you move
towards the lower right corner.

<div class="alert alert-block alert-danger">

What do you observe about the energies of the sub-images as frequencies
increase?</div>


In [ ]:
Yr = regroup(Y, N) / N
size_r = X.shape[0] // N
size_c = X.shape[1] // N

energy_map = np.zeros((N, N))
for row in range(N):
    for col in range(N):
        sub = Yr[row*size_r:(row+1)*size_r, col*size_c:(col+1)*size_c]
        energy_map[row, col] = np.sum(sub**2)

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(10*np.log10(energy_map + 1e-12), cmap='hot')
ax.set(xlabel='horizontal frequency index', ylabel='vertical frequency index',
       title='DCT sub-image energy (dB)')
fig.colorbar(im, ax=ax, shrink=0.8, label='dB')
fig.tight_layout()
fig.savefig('zach/images/dct_energy_heatmap.png', dpi=150, bbox_inches='tight')

print(f'DC (0,0): {energy_map[0,0]:.0f}')
print(f'(7,7):    {energy_map[7,7]:.0f}')
print(f'Ratio:    {energy_map[0,0]/energy_map[7,7]:.0f}x')

Now check that you can recover the original image from Y
by carrying out the inverse transform using:


In [ ]:
Z = colxfm(colxfm(Y.T, C8.T).T, C8.T)

fig, ax = plt.subplots()
plot_image(Z, ax=ax);

**Measure the maximum absolute error between X and Z
to confirm this.**


In [ ]:
print(f"Max absolute error X vs Z: {np.max(np.abs(X - Z)):.2e}")

The DCT analyses each $8 \times 8$ block of image pixels into a linear
combination of sixty-four $8 \times 8$ basis functions. The following will generate
an image comprising these basis functions (the `np.nan`s separate the sub-images as matplotlib draws them as transparent, and the `reshape` function converts from a matrix to a row vector):


In [ ]:
import numpy as np
# Stack some NaNs
bases = np.concatenate([np.full((8, 1), np.nan), C8, np.full((8, 1), np.nan)], axis=1)
# Reshape
bases_flat = np.reshape(bases, (-1, 1))

fig, ax = plt.subplots()
im = plot_image(255*bases_flat@bases_flat.T, ax=ax)
fig.colorbar(im);
fig.savefig('zach/images/dct_basis.png', dpi=150, bbox_inches='tight')

<div class="alert alert-block alert-danger">

Explain how this image relates to the DCT coefficients.</div>


The code constructs every 2-D basis function of the $8 \times 8$ DCT via an outer product of the 1-D basis vectors (basically for row transform, each row of the C8 matrix)

**Construction:**

1. `C8` is $8 \times 8$, where row $k$ is the 1-D basis vector $\mathbf{c}_k = [C_{k0},\, C_{k1},\, \ldots,\, C_{k7}]$. A NaN column is prepended and appended as a transparent separator, giving an $8 \times 10$ matrix.

2. This is reshaped into a column vector $\mathbf{b}$ of length 80, stacking all 8 basis rows end-to-end with NaN gaps between them.

3. The outer product $\mathbf{b}\,\mathbf{b}^T$ produces an $80 \times 80$ image. Within each non-NaN block at grid position $(k, l)$, the pixel at $(n_1, n_2)$ has value:

$$
C_{k}(n_1) \cdot C_{l}(n_2)
$$

This is exactly the $(k,l)$-th 2-D DCT basis function. The inverse DCT reconstructs a pixel block as a weighted sum of all 64 such basis functions:

$$
X(n_1, n_2) = \sum_{k=0}^{7}\sum_{l=0}^{7} Y(k,l)\; C_k(n_1)\; C_l(n_2)
$$

So each $8\times 8$ patch in the image is one of the 64 basis patterns that the DCT decomposes a pixel block into. The top-left patch is the DC (constant) basis, and frequency increases towards the bottom-right — horizontally along columns, vertically along rows. The DCT coefficient $Y(k,l)$ tells you _how much_ of the corresponding basis pattern is present in that block.


> is the whole point of this exercise to quantize in the freq domain rather than image domain
> Yes, exactly. The core idea is:

Direct quantisation (image domain): quantise pixels directly — every pixel gets the same coarse treatment, so you need many bits to represent the result since pixel values are spread across a wide range.

DCT then quantise (frequency domain): the DCT compacts most of the image energy into a few low-frequency coefficients. The high-frequency sub-images are mostly near zero. So when you quantise with the same step size, most high-frequency coefficients round to zero — far fewer bits needed via entropy coding.

The key insight is that the DCT doesn't discard anything by itself (it's perfectly invertible). The compression gain comes from the fact that quantisation in the transform domain is much more efficient because energy compaction means most coefficients are small/zero, giving peaked distributions with low entropy. In the pixel domain, values are spread out more uniformly, so entropy per pixel is higher.

That's also why dctbpp < bpp — each sub-image has a different (often very peaked) distribution, and coding them separately exploits this.


## 7.3 Quantisation and Coding Efficiency

We are now going to look at the effects of quantising the DCT coefficients
fairly coarsely and determine the entropies of the coefficient sub-images.
At this stage we shall quantise all sub-images with the same step-size,
since they all are the same size and have unit energy gain from the
quantiser to the output image (due to the orthonormal transform matrices).

First quantise the transformed image Y using a step size
of 17 to give Yq. Then regroup Yq to form
sub-images of each coefficient type as before, to give Yr. These sub-images have different probability distributions and we can take advantage of this later in coding them efficiently. Hence we get a better estimate of the number of bits required to code Yq by looking at the entropies of each of the re-grouped sub-images separately.

**Write a function `dctbpp(Yr, N)` to calculate the total number of bits from a re-grouped image Yr, by using `bpp(Ys)` on each sub-image Ys of Yr, then multiplying each result by the number of pixels in the sub-image, and summing to give the total number of bits.**


In [ ]:
from cued_sf2_lab.laplacian_pyramid import bpp

def dctbpp(Yr, N):
    """calculate total number of bits after dct
    take the regrouped image after dct (so each regroup is for a matrix basis)
    use N to break it into subimages
    do bpp on each subimage
    multiply by size to get total number of bits used"""
    bits_total = 0
    size_r = Yr.shape[0] // N
    size_c = Yr.shape[1] // N
    for row in range(N):
        for col in range(N):
            Ys = Yr[row*size_r:(row+1)*size_r, col*size_c:(col+1)*size_c]
            bits_total += bpp(Ys) * Ys.size
    return bits_total

<div class="alert alert-block alert-danger">

Visualise Yr and comment on the distributions in each of the sub-images. Use the function `dctbpp(Yr, N)` that you have written to calculate the total number of bits, and compare it with just using `bpp(Yr)`, explaining your results.

</div>


In [ ]:
from cued_sf2_lab.laplacian_pyramid import quantise

Yq = quantise(Y, 17)
Yr = regroup(Yq, N)

fig, ax = plt.subplots()
plot_image(Yr, ax=ax);

In [ ]:
print(f"bits using bpp(Yr) on whole image: {bpp(Yr) * Yr.size:.0f}")
print(f"bits using dctbpp(Yr, N) per sub-image: {dctbpp(Yr, N):.0f}")

`dctbpp` gives fewer bits than `bpp` on the whole regrouped image because it exploits the different probability distributions of each sub-image separately. The high-frequency sub-images are strongly peaked around zero (low entropy), while `bpp(Yr)` lumps all coefficient types into one histogram, blurring the peaked distributions into a broader one with higher entropy.


<div class="alert alert-block alert-danger">

Now reconstruct the output image `Z` from `Yq` and measure the rms
error (standard deviation) between `X` and `Z`. Compare this with the
error produced by quantising `X` with a step-size of 17 to give `Xq`.

</div>


In [ ]:
Z = colxfm(colxfm(Yq.T, C8.T).T, C8.T)
rms_dct_17 = np.std(X - Z)
rms_direct_17 = np.std(X - quantise(X, 17))
print(f"RMS error, DCT quantised at step=17:    {rms_dct_17:.3f}")
print(f"RMS error, direct quantisation step=17:  {rms_direct_17:.3f}")

The DCT gives a lower RMS error than direct quantisation at the same step size. This is because the orthonormal DCT preserves total quantisation noise energy, but concentrates most of the _signal_ energy into a few coefficients — so most coefficients are small and quantise to zero, contributing no noise at all. Direct quantisation adds noise uniformly to every pixel.


**_As with the Laplacian Pyramid, we really need to contrast compression ratios and visual results on compressed images with the same rms error. Re-use your step optimisation code to calculate the (non-integer) step size required in this case for the same rms error as quantising X with a step-size of 17._**


In [ ]:
step_grid = np.arange(1, 40, 0.1)
rms_direct = np.std(X - quantise(X, 17))

rms_vals = []
for s in step_grid:
    Yq_s = quantise(Y, s)
    Z_s = colxfm(colxfm(Yq_s.T, C8.T).T, C8.T)
    rms_vals.append(np.std(X - Z_s))
rms_vals = np.array(rms_vals)

best_step = step_grid[np.argmin(np.abs(rms_vals - rms_direct))]
Yq_matched = quantise(Y, best_step)
Z_matched = colxfm(colxfm(Yq_matched.T, C8.T).T, C8.T)
rms_matched = np.std(X - Z_matched)

print(f'Direct quantisation RMS: {rms_direct:.3f}')
print(f'DCT matched step: {best_step:.1f}, RMS: {rms_matched:.3f}')

<div class="alert alert-block alert-danger">

Calculate the compression ratio for this scheme compared to direct quantisation. Use `dctbpp` to calculate the number of bits needed. Contrast the visual appearance of the DCT-compressed image, the directly quantised image, and the original image.

</div>


In [ ]:
bits_direct = bpp(quantise(X, 17)) * X.size
bits_dct = dctbpp(regroup(Yq_matched, N), N)
CR = bits_direct / bits_dct
print(f'Direct quantisation: {bits_direct:.0f} bits')
print(f'DCT (8x8, step={best_step:.1f}): {bits_dct:.0f} bits')
print(f'Compression ratio: {CR:.3f}')

fig, axs = plt.subplots(1, 3, figsize=(12, 4))
plot_image(X, ax=axs[0])
axs[0].set(title='Original', xticks=[], yticks=[])

plot_image(quantise(X, 17), ax=axs[1])
axs[1].set(title=f'Direct quantise (step=17)\nRMS={rms_direct:.2f}', xticks=[], yticks=[])

plot_image(Z_matched, ax=axs[2])
axs[2].set(title=f'DCT 8x8 (step={best_step:.1f}), CR={CR:.2f}\nRMS={rms_matched:.2f}', xticks=[], yticks=[])

fig.tight_layout()
fig.savefig('zach/images/dct_visual_compare.png', dpi=150, bbox_inches='tight')

After regrouping, each sub-image collects a single coefficient type (e.g. all DC coefficients, all (0,1) coefficients) from every $8 \times 8$ block in the image. The plots below summarise the statistics across all $32 \times 32 = 1024$ blocks: total energy per coefficient type, fraction of coefficients zeroed by quantisation, and how energy is redistributed.


In [ ]:
# Energy spectrum of DCT coefficients and the effect of quantisation
Yr_raw = regroup(Y, N) / N          # unquantised, display-scaled
Yr_q   = regroup(Yq_matched, N) / N # quantised at matched step

size_r, size_c = X.shape[0] // N, X.shape[1] // N
energy_raw  = np.zeros((N, N))
energy_q    = np.zeros((N, N))
frac_zero   = np.zeros((N, N))

# Use unscaled for exact zero check
Yr_q_int = regroup(Yq_matched, N)
for r in range(N):
    for c in range(N):
        sub_raw = Yr_raw[r*size_r:(r+1)*size_r, c*size_c:(c+1)*size_c]
        sub_q   = Yr_q[r*size_r:(r+1)*size_r, c*size_c:(c+1)*size_c]
        sub_int = Yr_q_int[r*size_r:(r+1)*size_r, c*size_c:(c+1)*size_c]
        energy_raw[r, c] = np.sum(sub_raw**2)
        energy_q[r, c]   = np.sum(sub_q**2)
        frac_zero[r, c]  = np.mean(sub_int == 0)

fig, axs = plt.subplots(1, 3, figsize=(14, 4))

im0 = axs[0].imshow(10*np.log10(energy_raw + 1e-12), cmap='hot')
axs[0].set(title='Log energy per DCT band (dB)', xlabel='horiz freq', ylabel='vert freq')
fig.colorbar(im0, ax=axs[0], shrink=0.8)

im1 = axs[1].imshow(frac_zero * 100, cmap='Blues', vmin=0, vmax=100)
axs[1].set(title=f'% coeffs zeroed (step={best_step:.1f})', xlabel='horiz freq', ylabel='vert freq')
fig.colorbar(im1, ax=axs[1], shrink=0.8, label='%')

# 1-D view: flatten 2D freq index by radial distance from DC
radial = np.sqrt(np.arange(N)[:,None]**2 + np.arange(N)[None,:]**2).ravel()
order = np.argsort(radial)
e_raw_flat = energy_raw.ravel()[order]
e_q_flat   = energy_q.ravel()[order]
axs[2].semilogy(e_raw_flat, 'k-o', ms=3, label='before quantisation')
axs[2].semilogy(e_q_flat,   'r-s', ms=3, label='after quantisation')
axs[2].set(xlabel='DCT band (sorted by radial freq)', ylabel='energy',
           title='Energy compaction & quantisation')
axs[2].legend()
axs[2].grid(True)

fig.tight_layout()
fig.savefig('zach/images/dct_energy_spectrum.png', dpi=150, bbox_inches='tight')

We can also look at a single $8 \times 8$ block directly from the DCT output `Y` (before regrouping) to see the actual 64 coefficient values for one spatial patch, and how quantisation snaps the small high-frequency ones to zero.


In [ ]:
br, bc = 8, 8
block_raw = Y[br:br+N, bc:bc+N]
block_q   = Yq_matched[br:br+N, bc:bc+N]

fig, axs = plt.subplots(1, 3, figsize=(14, 4))
vmax = np.max(np.abs(block_raw))

im0 = axs[0].imshow(block_raw, cmap='RdBu_r', vmin=-vmax, vmax=vmax, interpolation='nearest')
axs[0].set(title='Before quantisation', xticks=range(N), yticks=range(N))
fig.colorbar(im0, ax=axs[0], shrink=0.8)

im1 = axs[1].imshow(block_q, cmap='RdBu_r', vmin=-vmax, vmax=vmax, interpolation='nearest')
axs[1].set(title=f'After quantisation (step={best_step:.1f})', xticks=range(N), yticks=range(N))
fig.colorbar(im1, ax=axs[1], shrink=0.8)

diff = block_raw - block_q
im2 = axs[2].imshow(diff, cmap='RdBu_r', vmin=-np.max(np.abs(diff)), vmax=np.max(np.abs(diff)),
                     interpolation='nearest')
axs[2].set(title='Quantisation error', xticks=range(N), yticks=range(N))
fig.colorbar(im2, ax=axs[2], shrink=0.8)

fig.suptitle(f'Single 8x8 DCT block at ({br},{bc}): {int(np.sum(block_q == 0))}/64 coefficients zeroed',
             fontsize=11)
fig.tight_layout()
fig.savefig('zach/images/dct_block_quantise.png', dpi=150, bbox_inches='tight')

### DCT energy spectrum and quantisation

**Left (energy heatmap):** The DC band $(0,0)$ sits at 63 dB, dropping to 28 dB at the high-frequency corner — a 35 dB ($\sim$3000$\times$) range. This is energy compaction: the DCT packs most of the image's energy into the few lowest-frequency bands.

**Middle (% zeroed):** Quantisation selectively zeros out high-frequency coefficients whose values are already small. The DC band has $\sim$0% zeros, while the highest-frequency bands reach 20–40%. The percentages are moderate (not 90%+) because the Lighthouse image has significant edge content (fence, lighthouse structure) that puts real energy into those bands.

**Right (energy before/after quantisation):** The two curves nearly overlap — quantisation at the matched step barely changes the total energy. This is the key mechanism: the quantiser spends its "noise budget" on bands that contribute almost nothing to perceived image quality, while the dominant low-frequency coefficients pass through nearly unaltered. This is why DCT + quantise outperforms direct pixel quantisation at equal RMS error.


## 7.4 Alternative transform sizes

So far, we have concentrated on $8 \times 8$ DCTs using C8
as the 1-D transform matrix. **Now generate 4-point and 16-point
transform matrices, C4 and C16 using `dct_ii`.**


In [ ]:
C2 = dct_ii(2)
C4 = dct_ii(4)
C16 = dct_ii(16)
C32 = dct_ii(32)

<div class="alert alert-block alert-danger">

Repeat the main measurements from the previous section, so as to obtain
estimates of the number of bits and compression ratios for $4 \times 4$ and $16 \times 16$ DCTs when the
rms errors are equivalent to those in your previous tests. Also assess the
relative subjective quality of the reconstructed images.</div>


In [ ]:
step_grid = np.arange(1, 50, 0.1)
results = {}
block_sizes = [2, 4, 8, 16, 32]

for Nblk in block_sizes:
    CN = dct_ii(Nblk)
    Y_N = colxfm(colxfm(X, CN).T, CN).T

    rms_vals = []
    for s in step_grid:
        Yq_s = quantise(Y_N, s)
        Z_s = colxfm(colxfm(Yq_s.T, CN.T).T, CN.T)
        rms_vals.append(np.std(X - Z_s))
    rms_vals = np.array(rms_vals)

    best_s = step_grid[np.argmin(np.abs(rms_vals - rms_direct))]
    Yq_s = quantise(Y_N, best_s)
    Z_s = colxfm(colxfm(Yq_s.T, CN.T).T, CN.T)
    rms_s = np.std(X - Z_s)
    bits_s = dctbpp(regroup(Yq_s, Nblk), Nblk)
    cr_s = bits_direct / bits_s

    results[Nblk] = dict(step=best_s, rms=rms_s, bits=bits_s, cr=cr_s, Z=Z_s)
    print(f'N={Nblk:2d}: step={best_s:.1f}, RMS={rms_s:.3f}, bits={bits_s:.0f}, CR={cr_s:.3f}')

# Full-image comparison: original + 5 sizes in a row
fig, axs = plt.subplots(1, 6, figsize=(18, 3))
plot_image(X, ax=axs[0])
axs[0].set(title='Original', xticks=[], yticks=[])

for ax, Nblk in zip(axs[1:], block_sizes):
    r = results[Nblk]
    plot_image(r['Z'], ax=ax)
    ax.set(title=f'{Nblk}×{Nblk}, CR={r["cr"]:.2f}', xticks=[], yticks=[])

fig.tight_layout()
fig.savefig('zach/images/dct_transform_sizes.png', dpi=150, bbox_inches='tight')

### Transform size comparison

At equal RMS error, $8 \times 8$ gives the best CR (2.94), slightly ahead of $16 \times 16$ (2.88), with $4 \times 4$ worst (2.65). The $4 \times 4$ blocks are too small to decorrelate much spatial redundancy. The $16 \times 16$ DCT doesn't beat $8 \times 8$ for two reasons: (1) the larger blocks straddle disparate content (e.g. fence next to sky), reducing the energy compaction benefit; (2) the `dctbpp` bias noted in the handout — with $N=16$, entropy is estimated over 256 sub-images of only $16 \times 16$ pixels each, making the bit estimate less reliable.

Visually, larger blocks produce more prominent **block artefacts** at boundaries since each block is quantised independently. The zoomed crop below shows this trade-off.


This analysis is in fact slightly biased because with larger transform sizes the function `dctbpp(Yr, N)` will use a greater number of smaller sub-images on which to calculate probability distributions. It may be better to use the same N in this function even when the actual transform changes; however whether this is more predictive of actual coding performance depends on what scanning method is used in the coding scheme.

<div class="alert alert-block alert-danger">

What happens in the limit if you use `dctbpp(Yr, 256)` (i.e. the entropy is calculated independently for each pixel)? Why is this the case, and why isn't this a realistic result?</div>


In [ ]:
Yq_17 = quantise(Y, 17)

bits_N8   = dctbpp(regroup(Yq_17, 8), 8)
bits_N256 = dctbpp(regroup(Yq_17, 8), 256)
bits_flat = bpp(regroup(Yq_17, 8)) * X.size

print(f'dctbpp with N=8   (64 sub-images):    {bits_N8:.0f} bits')
print(f'dctbpp with N=256 (65536 sub-images):  {bits_N256:.0f} bits')
print(f'bpp on whole image (1 distribution):   {bits_flat:.0f} bits')
print(f'Ratio N=8 / N=256: {bits_N8 / bits_N256:.2f}x')

`dctbpp(Yr, 256)` returns exactly 0 bits (and the ratio is $\infty$) because each "sub-image" is a single pixel. The entropy of a one-sample distribution is always zero: one value with probability 1 gives $-1 \cdot \log_2(1) = 0$. This is mathematically correct but completely unrealistic — you cannot actually encode an image with 0 bits. In practice, the codebook overhead for $256^2 = 65536$ separate single-sample distributions would far exceed any savings. Real entropy coders (Huffman, arithmetic) share a single code across many pixels of the same coefficient type, which is what `dctbpp(Yr, 8)` approximates.


<div class="alert alert-block alert-danger">

Can you draw any conclusions about the best choice of transform size for the
Lighthouse image? Try to postulate what features in other images might make your
conclusions different, and suggest why.</div>


In [ ]:
# Zoomed crop: original + 5 sizes in a row
r0, r1, c0, c1 = 100, 180, 50, 180

fig, axs = plt.subplots(1, 6, figsize=(18, 3))
plot_image(X[r0:r1, c0:c1], ax=axs[0])
axs[0].set(title='Original (crop)', xticks=[], yticks=[])

for ax, Nblk in zip(axs[1:], block_sizes):
    r = results[Nblk]
    plot_image(r['Z'][r0:r1, c0:c1], ax=ax)
    ax.set(title=f'{Nblk}×{Nblk}, CR={r["cr"]:.2f}', xticks=[], yticks=[])

fig.tight_layout()
fig.savefig('zach/images/dct_crop_compare.png', dpi=150, bbox_inches='tight')

### Conclusions on transform size

For the Lighthouse image, $8 \times 8$ is the best practical choice. The $16 \times 16$ DCT achieves a slightly higher CR due to better energy compaction across larger blocks, but at the cost of more visible block artefacts — the block boundaries become prominent in smooth regions like the sky. The $4 \times 4$ DCT has the worst CR because the blocks are too small to capture spatial correlations, but its artefacts are the least noticeable since the block boundaries are finer.

**Image-dependent factors:**

- Images with large smooth regions (sky, gradients) benefit from larger $N$ — more spatial correlation to exploit, and the block artefacts fall in perceptually less sensitive areas.
- Images with dense fine detail or many edges (textures, text) favour smaller $N$ — the edges already break spatial correlation across large blocks, so the larger DCT gains little compaction but introduces conspicuous blocking at every boundary.
- The $8 \times 8$ size used by JPEG is a well-known compromise between these extremes.
